# INEN energy demand calibration to IEA — Morocco

Calibrates industrial (INEN) energy demand against **IEA Industry Total Final Consumption by source** (`iea_morocco_industry_tfc_by_source.csv`). Two stages:

- **Stage 0 — total**: scale `consumpinit_inen_*` so the model's total INEN energy demand matches the IEA industry total at CALIB_YEAR.
- **Stage B — mix**: set INEN fuel fractions so the fuel split matches the IEA mix (by IEA fuel group). All INEN industries get the IEA national-average mix because IEA provides no per-industry breakdown.

SCOE fuel-mix overrides are kept as a separate editable block (no IEA SCOE data available).

IEA fuel groups -> SISEPUEDE fuels: Coal group = `coal`; Oil group = `oil`+`diesel`+`gasoline`+`kerosene`+`hydrocarbon_gas_liquids`+`coke` (petcoke); Natural gas = `natural_gas`; Electricity = `electricity`; Biofuels & waste = `biomass`.

In [ ]:
# ============================================================
# Parameters
# ============================================================
BASE_INPUT_CSV = "../../input_data/sisepuede_raw_input_morocco_fuels.csv"
OUT_INPUT_CSV  = "../../input_data/sisepuede_raw_input_morocco_fuels.csv"

# IEA industry total final consumption by source (TJ), 2000-2023
IEA_INDUSTRY_CSV = "../../input_data/reference/iea_morocco_industry_tfc_by_source.csv"

CALIB_YEAR = 2022
BASE_YEAR = 2015
TERMINAL_YEAR = 2050
REGION = "morocco"

# Year from which fuel-mix overrides apply (BASE_YEAR -> whole series, no 2022 step)
OVERRIDE_FROM_YEAR = BASE_YEAR

# Stage 0 convergence
TOTAL_REL_TOLERANCE = 0.01     # accept INEN total within 1% of IEA
MAX_TOTAL_ITER = 4             # consumpinit-scaling iterations

# Per-(sector,fuel) diagnostic tolerance
REL_TOLERANCE = 0.10

# ------------------------------------------------------------
# Sub-split of the IEA "Oil and oil products" group across SISEPUEDE fuels.
# IEA does not break this down; these are best estimates for Moroccan industry.
#   coke   = petroleum coke (cement kilns - dominant petcoke user)
#   oil    = heavy fuel oil (industrial boilers, OCP)
#   diesel = mining haul trucks, heavy machinery
# Must sum to 1.0.
OIL_GROUP_SUBSPLIT = {
    "coke":                     0.35,
    "oil":                      0.42,
    "diesel":                   0.18,
    "kerosene":                 0.02,
    "gasoline":                 0.02,
    "hydrocarbon_gas_liquids":  0.01,
}

# ------------------------------------------------------------
# SCOE fuel-mix overrides (heat fractions). No IEA SCOE data -> Morocco best estimates.
# {(scoe_category, fuel): fraction}. Each category must sum to <= 1.
SCOE_OVERRIDES = {
    ("residential", "hydrocarbon_gas_liquids"): 0.30,
    ("residential", "biomass"):                  0.20,
    ("residential", "electricity"):              0.40,
    ("residential", "natural_gas"):              0.05,
    ("residential", "kerosene"):                 0.03,
    ("commercial_municipal", "hydrocarbon_gas_liquids"): 0.40,
    ("commercial_municipal", "natural_gas"):              0.10,
    ("commercial_municipal", "electricity"):              0.40,
    ("commercial_municipal", "diesel"):                   0.07,
    ("commercial_municipal", "biomass"):                  0.03,
    ("other_se", "hydrocarbon_gas_liquids"): 0.25,
    ("other_se", "natural_gas"):              0.05,
    ("other_se", "electricity"):              0.50,
    ("other_se", "diesel"):                   0.10,
    ("other_se", "biomass"):                  0.10,
}


In [ ]:
import os, sys, pathlib, warnings, logging
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

HERE = pathlib.Path(os.getcwd())
sys.path.insert(0, str(HERE.parent))
from utils.logger_utils import setup_clean_logger, mute_external_loggers
logger = setup_clean_logger("recalib_inen_iea", logging.INFO)
mute_external_loggers(["sisepuede"])

TJ_TO_PJ = 1e-3

# SISEPUEDE INEN fuel -> IEA fuel group
SSP_FUEL_TO_IEA_GROUP = {
    "coal":                     "Coal and coal products",
    "furnace_gas":              "Coal and coal products",
    "coke":                     "Oil and oil products",   # petcoke
    "oil":                      "Oil and oil products",
    "diesel":                   "Oil and oil products",
    "gasoline":                 "Oil and oil products",
    "kerosene":                 "Oil and oil products",
    "hydrocarbon_gas_liquids":  "Oil and oil products",
    "natural_gas":              "Natural gas",
    "electricity":              "Electricity",
    "biomass":                  "Biofuels and waste",
    "hydrogen":                 "Electricity",            # negligible 2022
    "solar":                    "Electricity",            # negligible 2022
}
IEA_GROUPS = ["Coal and coal products", "Oil and oil products",
              "Natural gas", "Electricity", "Biofuels and waste"]


## 1. Helpers

In [ ]:
import sisepuede.core.attribute_table as att
import sisepuede.manager.sisepuede_examples as sxl
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
from ssp_transformations_handler.GeneralUtils import GeneralUtils

_EX = sxl.SISEPUEDEExamples()


def run_energy_only(csv_path, y0=BASE_YEAR, y1=2050):
    """Run EnergyConsumption only (no NemoMod)."""
    df_raw = pd.read_csv(csv_path)
    fs = sfs.SISEPUEDEFileStructure(initialize_directories=False)
    kt, ky = fs.model_attributes.dim_time_period, fs.model_attributes.field_dim_year
    years = np.arange(y0, y1 + 1).astype(int)
    fs.model_attributes.update_dimensional_attribute_table(
        att.AttributeTable(pd.DataFrame({kt: range(len(years)), ky: years}), kt))
    models = sm.SISEPUEDEModels(
        fs.model_attributes, allow_electricity_run=False,
        fp_julia=fs.dir_jl, fp_nemomod_reference_files=fs.dir_ref_nemo,
        initialize_julia=False)
    df = GeneralUtils().add_missing_cols(_EX("input_data_frame"), df_raw.copy())
    df["region"] = REGION
    df = df[df["year"] <= y1]
    o = models.project(df, include_electricity_in_energy=False)
    o["year"] = o["time_period"].apply(lambda t: y0 + int(t))
    return o


def inen_demand_by_fuel(df_out, year):
    """Per-fuel INEN demand (PJ) at year, dict fuel->PJ."""
    row = df_out[df_out["year"] == year].iloc[0]
    pref = "energy_demand_enfu_subsector_total_pj_inen_fuel_"
    return {c[len(pref):]: float(row[c]) for c in df_out.columns if c.startswith(pref)}


def inen_demand_by_iea_group(df_out, year):
    """Aggregate INEN per-fuel demand into IEA fuel groups (PJ)."""
    by_fuel = inen_demand_by_fuel(df_out, year)
    grp = {g: 0.0 for g in IEA_GROUPS}
    for fuel, pj in by_fuel.items():
        g = SSP_FUEL_TO_IEA_GROUP.get(fuel)
        if g:
            grp[g] += pj
    grp["TOTAL"] = sum(grp[g] for g in IEA_GROUPS)
    return grp


def scoe_demand_by_fuel(df_out, year):
    row = df_out[df_out["year"] == year].iloc[0]
    pref = "energy_demand_enfu_subsector_total_pj_scoe_fuel_"
    return {c[len(pref):]: float(row[c]) for c in df_out.columns if c.startswith(pref)}


## 2. Load IEA industry targets

Reads the IEA industry TFC-by-source CSV and extracts the CALIB_YEAR targets per fuel group.

In [ ]:
iea_raw = pd.read_csv(IEA_INDUSTRY_CSV)
iea_raw.columns = [c.strip().strip('"') for c in iea_raw.columns]
src_col = iea_raw.columns[0]   # the source-name column
iea_raw = iea_raw.rename(columns={src_col: "source"})
iea_raw["source"] = iea_raw["source"].str.strip().str.strip('"')
iea_raw["Value"] = iea_raw["Value"].astype(float)
iea_raw["Year"] = iea_raw["Year"].astype(int)

iea_calib = iea_raw[iea_raw["Year"] == CALIB_YEAR].copy()
iea_calib["PJ"] = iea_calib["Value"] * TJ_TO_PJ

IEA_INEN_TARGETS_PJ = dict(zip(iea_calib["source"], iea_calib["PJ"]))
IEA_INEN_TOTAL_PJ = float(iea_calib["PJ"].sum())

print(f"IEA Industry TFC by source, Morocco {CALIB_YEAR}:")
for g in IEA_GROUPS:
    pj = IEA_INEN_TARGETS_PJ.get(g, 0.0)
    print(f"  {g:28s} {pj:8.2f} PJ  ({100*pj/IEA_INEN_TOTAL_PJ:5.1f}%)")
print(f"  {'TOTAL':28s} {IEA_INEN_TOTAL_PJ:8.2f} PJ")


## 3. Stage 0 — Calibrate INEN total demand to IEA

Runs the energy model, measures total INEN demand, and scales every `consumpinit_inen_*` column by `IEA_total / model_total`. Iterates until the model total is within `TOTAL_REL_TOLERANCE` of the IEA total. The scaling factor is applied uniformly across all time periods and all industries (preserves the relative size of each industry).

In [ ]:
df_in = pd.read_csv(BASE_INPUT_CSV)
consumpinit_cols = [c for c in df_in.columns if c.startswith("consumpinit_inen_energy_")]
print(f"Found {len(consumpinit_cols)} consumpinit_inen_energy_* columns to scale")

# Iterative scaling
import tempfile
tmp_csv = OUT_INPUT_CSV.replace(".csv", "_stage0_tmp.csv")
cumulative_factor = 1.0
for it in range(1, MAX_TOTAL_ITER + 1):
    df_in.to_csv(tmp_csv, index=False)
    o = run_energy_only(tmp_csv)
    grp = inen_demand_by_iea_group(o, CALIB_YEAR)
    model_total = grp["TOTAL"]
    rel = abs(model_total - IEA_INEN_TOTAL_PJ) / IEA_INEN_TOTAL_PJ
    print(f"  iter {it}: model INEN total = {model_total:.2f} PJ  "
          f"(IEA {IEA_INEN_TOTAL_PJ:.2f}, gap {100*rel:+.2f}%)")
    if rel <= TOTAL_REL_TOLERANCE:
        print(f"  converged within {TOTAL_REL_TOLERANCE*100:.0f}% tolerance")
        break
    factor = IEA_INEN_TOTAL_PJ / model_total
    cumulative_factor *= factor
    for c in consumpinit_cols:
        df_in[c] = df_in[c] * factor

print(f"\nCumulative consumpinit_inen scaling factor: {cumulative_factor:.4f}")
import os as _os
if _os.path.exists(tmp_csv):
    _os.remove(tmp_csv)
# df_in now holds the total-calibrated data (consumpinit scaled). Mix applied in Stage B.


## 4. Stage A — Diagnostic: INEN by IEA fuel group (post total-calibration, pre-mix)

Shows where the current fuel mix sits versus the IEA group targets, after the total has been calibrated. The gaps here are what Stage B corrects.

In [ ]:
df_in.to_csv(OUT_INPUT_CSV, index=False)
o_pre = run_energy_only(OUT_INPUT_CSV)
grp_pre = inen_demand_by_iea_group(o_pre, CALIB_YEAR)

print(f"=" * 70)
print(f"INEN by IEA fuel group at {CALIB_YEAR} — model vs IEA (post Stage 0)")
print(f"=" * 70)
print(f"{'IEA group':28s} {'model':>10s} {'IEA':>10s} {'gap':>10s}  {'gap %':>8s}")
print("-" * 70)
for g in IEA_GROUPS:
    m = grp_pre[g]; t = IEA_INEN_TARGETS_PJ.get(g, 0.0)
    gap = m - t
    pct = 100 * gap / t if t > 0.01 else float("nan")
    print(f"{g:28s} {m:10.2f} {t:10.2f} {gap:+10.2f}  {pct:+7.1f}%")
print("-" * 70)
print(f"{'TOTAL':28s} {grp_pre['TOTAL']:10.2f} {IEA_INEN_TOTAL_PJ:10.2f} "
      f"{grp_pre['TOTAL']-IEA_INEN_TOTAL_PJ:+10.2f}")


## 5. Stage B — Apply IEA fuel mix to all INEN industries + SCOE overrides

Builds the IEA national-average INEN fuel mix (group shares from the IEA CSV, oil group sub-split from `OIL_GROUP_SUBSPLIT`) and applies it to **every** INEN industry. Because IEA gives no per-industry breakdown, the national mix is the maximum-information choice and it guarantees the INEN aggregate matches IEA. SCOE categories get `SCOE_OVERRIDES`.

In [ ]:
# Build IEA national INEN fuel mix (fraction per SISEPUEDE fuel)
tot = IEA_INEN_TOTAL_PJ
share_coal    = IEA_INEN_TARGETS_PJ.get("Coal and coal products", 0.0) / tot
share_oilgrp  = IEA_INEN_TARGETS_PJ.get("Oil and oil products", 0.0) / tot
share_ng      = IEA_INEN_TARGETS_PJ.get("Natural gas", 0.0) / tot
share_elec    = IEA_INEN_TARGETS_PJ.get("Electricity", 0.0) / tot
share_biomass = IEA_INEN_TARGETS_PJ.get("Biofuels and waste", 0.0) / tot

assert abs(sum(OIL_GROUP_SUBSPLIT.values()) - 1.0) < 1e-6, "OIL_GROUP_SUBSPLIT must sum to 1"

# IEA target FINAL-energy (fuel-demand) shares per SISEPUEDE fuel
IEA_FINAL_SHARE = {
    "coal":        share_coal,
    "natural_gas": share_ng,
    "electricity": share_elec,
    "biomass":     share_biomass,
}
for fuel, sub in OIL_GROUP_SUBSPLIT.items():
    IEA_FINAL_SHARE[fuel] = share_oilgrp * sub

# IMPORTANT: frac_inen_energy_* is a USEFUL-energy share; the model computes
# fuel_demand_f = useful_total * frac_f / efficfactor_f. IEA reports FINAL energy
# (= fuel demand). To hit a target final-energy share T_f, the fraction must be
# set to T_f * efficfactor_f, then normalized:
#     frac_f = (T_f * efficfactor_f) / sum_g(T_g * efficfactor_g)
inen_eff = {}
for fuel in IEA_FINAL_SHARE:
    eff_col = f"efficfactor_enfu_industrial_energy_fuel_{fuel}"
    assert eff_col in df_in.columns, f"missing {eff_col}"
    inen_eff[fuel] = float(df_in.loc[df_in["time_period"] == 0, eff_col].iloc[0])

_weights = {f: IEA_FINAL_SHARE[f] * inen_eff[f] for f in IEA_FINAL_SHARE}
_wsum = sum(_weights.values())
IEA_INEN_MIX = {f: w / _wsum for f, w in _weights.items()}

print(f"Fuel efficiency factors (efficfactor_enfu_industrial_energy_fuel_*):")
for f in sorted(inen_eff, key=lambda x: -inen_eff[x]):
    print(f"  {f:28s} eff={inen_eff[f]:.3f}  IEA_final_share={IEA_FINAL_SHARE[f]:.4f}")

print(f"\nEfficiency-corrected INEN useful-energy fractions ({CALIB_YEAR}):")
for f, v in sorted(IEA_INEN_MIX.items(), key=lambda x: -x[1]):
    print(f"  {f:28s} {v:.4f}")
print(f"  {'SUM':28s} {sum(IEA_INEN_MIX.values()):.4f}")

# Canonical fuel suffixes
INEN_FUEL_SUFFIXES = ["coal","coke","diesel","electricity","furnace_gas","gasoline",
                      "hydrocarbon_gas_liquids","hydrogen","kerosene","natural_gas",
                      "oil","solar","solid_biomass"]
SCOE_FUEL_SUFFIXES = ["coal","diesel","electricity","gasoline","hydrocarbon_gas_liquids",
                      "hydrogen","kerosene","natural_gas","solid_biomass"]
OUTPUT_TO_INPUT_FUEL = {"biomass": "solid_biomass"}
def input_fuel(f): return OUTPUT_TO_INPUT_FUEL.get(f, f)

def _extract_subcats(cols, prefix, suffixes):
    out = set()
    for c in cols:
        if not c.startswith(prefix): continue
        tail = c[len(prefix):]
        for s in suffixes:
            if tail.endswith("_" + s):
                out.add(tail[:-(len(s)+1)]); break
    return sorted(out)

INEN_SUBCATS = _extract_subcats(df_in.columns, "frac_inen_energy_", INEN_FUEL_SUFFIXES)
SCOE_SUBCATS = _extract_subcats(df_in.columns, "frac_scoe_heat_energy_", SCOE_FUEL_SUFFIXES)
print(f"\nINEN subcategories: {len(INEN_SUBCATS)}, SCOE subcategories: {len(SCOE_SUBCATS)}")

# Build FRAC_OVERRIDES: IEA mix for every INEN industry + SCOE_OVERRIDES
FRAC_OVERRIDES = {}
for sub in INEN_SUBCATS:
    for fuel, frac in IEA_INEN_MIX.items():
        FRAC_OVERRIDES[(sub, fuel)] = frac
FRAC_OVERRIDES.update(SCOE_OVERRIDES)
print(f"FRAC_OVERRIDES built: {len(FRAC_OVERRIDES)} entries "
      f"({len(INEN_SUBCATS)} INEN industries x {len(IEA_INEN_MIX)} fuels + {len(SCOE_OVERRIDES)} SCOE)")


In [ ]:
# Apply the overrides (grouped by subcategory, renormalize free fuels, hold to TERMINAL_YEAR)
from collections import defaultdict
tp_from = OVERRIDE_FROM_YEAR - BASE_YEAR
tp_terminal = TERMINAL_YEAR - BASE_YEAR

by_subcat = defaultdict(dict)
for (sub, fuel_out), frac in FRAC_OVERRIDES.items():
    by_subcat[sub][input_fuel(fuel_out)] = float(frac)

def find_frac_cols(subcat):
    if subcat in INEN_SUBCATS:
        pfx = f"frac_inen_energy_{subcat}_"
        return "INEN", pfx, [f"{pfx}{f}" for f in INEN_FUEL_SUFFIXES if f"{pfx}{f}" in df_in.columns]
    if subcat in SCOE_SUBCATS:
        pfx = f"frac_scoe_heat_energy_{subcat}_"
        return "SCOE", pfx, [f"{pfx}{f}" for f in SCOE_FUEL_SUFFIXES if f"{pfx}{f}" in df_in.columns]
    return None, None, []

print(f"Applying overrides across {len(by_subcat)} subcategories, tp {tp_from}-{tp_terminal}")
n_applied = 0
for subcat, fuel_to_frac in by_subcat.items():
    sector, pfx, all_cols = find_frac_cols(subcat)
    if not all_cols:
        print(f"  SKIP {subcat!r}: no frac columns"); continue
    overridden = {}
    for fuel_in, frac in fuel_to_frac.items():
        tcol = f"{pfx}{fuel_in}"
        if tcol in all_cols:
            overridden[tcol] = frac
    if not overridden: continue
    s_over = sum(overridden.values())
    if s_over > 1.0 + 1e-6:
        print(f"  ERROR {subcat!r}: overrides sum {s_over:.4f} > 1, skipping"); continue
    remaining = max(0.0, 1.0 - s_over)
    free_cols = [c for c in all_cols if c not in overridden]
    for tp in range(tp_from, tp_terminal + 1):
        mask = df_in["time_period"] == tp
        free_old = float(df_in.loc[mask, free_cols].sum(axis=1).iloc[0]) if free_cols else 0.0
        for c, v in overridden.items():
            df_in.loc[mask, c] = v
        if free_cols:
            if free_old > 1e-9:
                sc = remaining / free_old
                for c in free_cols:
                    df_in.loc[mask, c] = df_in.loc[mask, c] * sc
            else:
                for c in free_cols:
                    df_in.loc[mask, c] = remaining / len(free_cols)
    n_applied += 1
print(f"Applied {n_applied} subcategory groups")

df_in.to_csv(OUT_INPUT_CSV, index=False)
print(f"Wrote calibrated input: {OUT_INPUT_CSV}")


## 6. Validation — INEN by IEA fuel group after Stage 0 + Stage B

Re-runs the energy model on the final CSV and confirms the INEN total and the per-group split both match IEA.

In [ ]:
o_final = run_energy_only(OUT_INPUT_CSV)
grp_final = inen_demand_by_iea_group(o_final, CALIB_YEAR)

print(f"=" * 72)
print(f"VALIDATION — INEN by IEA fuel group at {CALIB_YEAR}")
print(f"=" * 72)
print(f"{'IEA group':28s} {'model':>10s} {'IEA':>10s} {'gap':>9s} {'gap %':>8s}  status")
print("-" * 72)
all_ok = True
for g in IEA_GROUPS:
    m = grp_final[g]; t = IEA_INEN_TARGETS_PJ.get(g, 0.0)
    gap = m - t
    pct = 100 * gap / t if t > 0.01 else float("nan")
    ok = (abs(gap) <= REL_TOLERANCE * t) if t > 0.01 else (abs(m) < 0.5)
    if not ok: all_ok = False
    print(f"{g:28s} {m:10.2f} {t:10.2f} {gap:+9.2f} {pct:+7.1f}%  {'OK' if ok else 'MISS'}")
print("-" * 72)
tot_gap = grp_final["TOTAL"] - IEA_INEN_TOTAL_PJ
print(f"{'TOTAL':28s} {grp_final['TOTAL']:10.2f} {IEA_INEN_TOTAL_PJ:10.2f} "
      f"{tot_gap:+9.2f} {100*tot_gap/IEA_INEN_TOTAL_PJ:+7.1f}%")
print(f"\n{'ALL GROUPS WITHIN TOLERANCE' if all_ok else 'SOME GROUPS OUT OF TOLERANCE'}")


## 7. Diagnostic plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: INEN by IEA group — model (pre/post) vs IEA
ax = axes[0]
x = np.arange(len(IEA_GROUPS)); w = 0.27
ax.bar(x - w, [grp_pre[g] for g in IEA_GROUPS], w, label="model pre-mix", color="lightcoral")
ax.bar(x,     [grp_final[g] for g in IEA_GROUPS], w, label="model calibrated", color="steelblue")
ax.bar(x + w, [IEA_INEN_TARGETS_PJ.get(g,0) for g in IEA_GROUPS], w, label="IEA target", color="orange")
ax.set_xticks(x)
ax.set_xticklabels([g.replace(" and ","/\n").replace(" ","\n") for g in IEA_GROUPS], fontsize=8)
ax.set_ylabel("INEN demand (PJ)")
ax.set_title(f"INEN by IEA fuel group @ {CALIB_YEAR}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# Right: IEA industry total time series vs model
ax = axes[1]
iea_ts = iea_raw.groupby("Year")["Value"].sum() * TJ_TO_PJ
ax.plot(iea_ts.index, iea_ts.values, "k^-", label="IEA industry total", alpha=0.7)
model_ts = [inen_demand_by_iea_group(o_final, y)["TOTAL"]
            for y in range(2018, 2051)]
ax.plot(range(2018, 2051), model_ts, "s-", label="model INEN total (calibrated)", color="steelblue")
ax.axvline(CALIB_YEAR, ls=":", c="red", alpha=0.5)
ax.set_xlabel("Year"); ax.set_ylabel("INEN total (PJ)")
ax.set_title("INEN total: model vs IEA")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Notes

- **Stage 0** scales every `consumpinit_inen_energy_*` column by a single cumulative factor so the model INEN total matches the IEA industry total. The relative size of each industry is preserved.
- **Stage B** applies the IEA national-average fuel mix to *every* INEN industry. IEA publishes industry energy only at the national level (no per-industry breakdown), so the national mix is applied uniformly. This guarantees the INEN aggregate matches IEA but means individual industries (e.g. cement) carry the national-average mix rather than a sector-specific one. If a future analysis needs per-industry realism, reintroduce variation while keeping the production-weighted aggregate equal to the IEA mix.
- **`coke` is treated as petroleum coke** and mapped to the IEA "Oil and oil products" group (Moroccan cement kilns burn petcoke). The split of the oil group across oil/coke/diesel/kerosene/gasoline/LPG is set by `OIL_GROUP_SUBSPLIT` (IEA gives no breakdown within the group) — tune it if better data appears.
- **SCOE** is calibrated separately via `SCOE_OVERRIDES` (no IEA SCOE balance available).
- Output CSV is written in place to `sisepuede_raw_input_morocco_fuels.csv`. A backup should be taken before running.